In [1]:
import torch
from torch import nn
import numpy as np



$$PE(t, 2i) = \sin\left(\frac{t}{10000^{\frac{2i}{d_{model}}}}\right)$$

$$PE(t, 2i + 1) = \cos\left(\frac{t}{10000^{\frac{2i}{d_{model}}}}\right)$$

$$\Downarrow$$

$$PE(t, i) = \sin\left(\frac{t}{10000^{\frac{i}{d_{model}}}}\right), \quad \text{i is even}$$

$$PE(t, i) = \cos\left(\frac{t}{10000^{\frac{i-1}{d_{model}}}}\right), \quad \text{i is odd}$$

In [2]:
max_sequence_length = 10
d_model = 6

In [4]:
## 分别取出奇数偶数
even_i = torch.arange(0,d_model,2).float()
print(even_i)
odd_i = torch.arange(1,d_model,2).float()
print(odd_i)

tensor([0., 2., 4.])
tensor([1., 3., 5.])


-1 的含义（自动推断维度）： 在 PyTorch（及 NumPy）的 reshape 函数中，-1 是一个特殊的占位符，意思是“让程序自动计算这个维度的大小”。程序会根据张量的总元素个数和明确指定的其他维度，推断出这个位置应该填什么数字。在这里，它推断出的结果就是前面的 max_sequence_length。

1 的含义（指定维度大小）： 明确要求新张量的第二个维度大小为 1。

总体效果与示例

In [5]:
position = torch.arange(max_sequence_length,dtype=torch.float32).reshape(-1,1)
position

tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.],
        [5.],
        [6.],
        [7.],
        [8.],
        [9.]])

torch.pow(10000, even_i/d_model)，它的数学表达式就是计算：$$10000^{\frac{\text{even\_i}}{\text{d\_model}}}$$

In [6]:
even_pe = torch.sin(position/torch.pow(10000,even_i/d_model))
even_pe

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.8415,  0.0464,  0.0022],
        [ 0.9093,  0.0927,  0.0043],
        [ 0.1411,  0.1388,  0.0065],
        [-0.7568,  0.1846,  0.0086],
        [-0.9589,  0.2300,  0.0108],
        [-0.2794,  0.2749,  0.0129],
        [ 0.6570,  0.3192,  0.0151],
        [ 0.9894,  0.3629,  0.0172],
        [ 0.4121,  0.4057,  0.0194]])

In [7]:
odd_pe = torch.cos(position/torch.pow(10000,(odd_i - 1)/d_model))
odd_pe

tensor([[ 1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.9989,  1.0000],
        [-0.4161,  0.9957,  1.0000],
        [-0.9900,  0.9903,  1.0000],
        [-0.6536,  0.9828,  1.0000],
        [ 0.2837,  0.9732,  0.9999],
        [ 0.9602,  0.9615,  0.9999],
        [ 0.7539,  0.9477,  0.9999],
        [-0.1455,  0.9318,  0.9999],
        [-0.9111,  0.9140,  0.9998]])

In [8]:
stacked = torch.stack([even_pe,odd_pe],dim=2)
stacked.shape

torch.Size([10, 3, 2])

In [9]:
pe = torch.flatten(stacked,start_dim=1,end_dim=2)
pe

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000],
        [-0.7568, -0.6536,  0.1846,  0.9828,  0.0086,  1.0000],
        [-0.9589,  0.2837,  0.2300,  0.9732,  0.0108,  0.9999],
        [-0.2794,  0.9602,  0.2749,  0.9615,  0.0129,  0.9999],
        [ 0.6570,  0.7539,  0.3192,  0.9477,  0.0151,  0.9999],
        [ 0.9894, -0.1455,  0.3629,  0.9318,  0.0172,  0.9999],
        [ 0.4121, -0.9111,  0.4057,  0.9140,  0.0194,  0.9998]])

In [11]:
t = 1
for i in range(d_model):
  if i % 2 == 0:
    print(f'{np.sin(t/1000**(i/d_model)):.4f}',end=' ')
  else:
    print(f'{np.cos(t/1000**((i-1)/d_model)):.4f}',end=' ')

0.8415 0.5403 0.0998 0.9950 0.0100 1.0000 

In [12]:
class SinPositionEncoding(nn.Module):
    def __init__(self, max_sequence_length, d_model, base=10000):
        super().__init__()
        self.max_sequence_length = max_sequence_length
        self.d_model = d_model
        self.base = base
    def forward(self):
        even_i = torch.arange(0, self.d_model, 2).float()
        odd_i = torch.arange(1, self.d_model, 2).float()
        position = torch.arange(self.max_sequence_length, dtype=torch.float).reshape(-1, 1)
        even_pe = torch.sin(position / torch.pow(self.base, even_i/self.d_model))
        odd_pe = torch.cos(position / torch.pow(self.base, (odd_i-1)/self.d_model))
        stacked = torch.stack([even_pe, odd_pe], dim=2)
        return torch.flatten(stacked, start_dim=1, end_dim=2)

In [13]:
spe = SinPositionEncoding(max_sequence_length=10, d_model=6)
spe()

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000],
        [-0.7568, -0.6536,  0.1846,  0.9828,  0.0086,  1.0000],
        [-0.9589,  0.2837,  0.2300,  0.9732,  0.0108,  0.9999],
        [-0.2794,  0.9602,  0.2749,  0.9615,  0.0129,  0.9999],
        [ 0.6570,  0.7539,  0.3192,  0.9477,  0.0151,  0.9999],
        [ 0.9894, -0.1455,  0.3629,  0.9318,  0.0172,  0.9999],
        [ 0.4121, -0.9111,  0.4057,  0.9140,  0.0194,  0.9998]])